In [0]:
%python
customers_bronze = spark.table(
    "workspace.banking_bronze.customers"
)

print("Bronze records:", customers_bronze.count())

In [0]:
%python
customers_bronze = spark.table(
    "workspace.banking_bronze.customers"
)

In [0]:
%python
print("Bronze records:", customers_bronze.count())

In [0]:
%python
customers_silver = customers_bronze.dropDuplicates(["customer_id"])

In [0]:
%python
print("Silver records:", customers_silver.count())

In [0]:
%python
customers_silver = customers_bronze.dropDuplicates(["customer_id"])

print("Bronze records:", customers_bronze.count())
print("Silver records:", customers_silver.count())

In [0]:
%python
from pyspark.sql.functions import col, sum

In [0]:
%python
customers_silver.select(
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("customer_name").isNull().cast("int")).alias("name_nulls"),
    sum(col("email").isNull().cast("int")).alias("email_nulls"),
    sum(col("age").isNull().cast("int")).alias("age_nulls"),
    sum(col("city").isNull().cast("int")).alias("city_nulls"),
    sum(col("state").isNull().cast("int")).alias("state_nulls"),
    sum(col("registration_date").isNull().cast("int")).alias("date_nulls")
).show()

In [0]:
%python
from pyspark.sql.functions import coalesce, lit
customers_silver = customers_silver.withColumn(
    "email",
    coalesce(col("email"), lit("Unknown"))
)

In [0]:
%python
customers_silver = customers_silver.withColumn(
    "city",
    coalesce(col("city"), lit("Unknown"))
)

In [0]:
%python
invalid_age_count = customers_silver.filter(
    (col("age") < 18) | (col("age") > 100)
).count()

print("Invalid age records:", invalid_age_count)

In [0]:
%python
customers_silver.show(10)

In [0]:
%python
customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.banking_silver.customers")

In [0]:
%python
spark.table("workspace.banking_silver.customers").count()

In [0]:
%python
accounts_bronze = spark.table(
    "workspace.banking_bronze.accounts"
)

print("Bronze account records:", accounts_bronze.count())

In [0]:
%python
accounts_bronze.show(5)

In [0]:
%python
print("Total records:", accounts_bronze.count())

print(
    "Unique account IDs:",
    accounts_bronze.select("account_id").distinct().count()
)

In [0]:
%python
accounts_silver = accounts_bronze.dropDuplicates(["account_id"])

print("Before:", accounts_bronze.count())
print("After :", accounts_silver.count())

In [0]:
%python
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
%python
print("Total records:", accounts_bronze.count())

print(
    "Unique account IDs:",
    accounts_bronze.select("account_id").distinct().count()
)

In [0]:
%python
accounts_silver = accounts_bronze.dropDuplicates(["account_id"])

print("Before:", accounts_bronze.count())
print("After :", accounts_silver.count())

In [0]:
%python
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
%python
accounts_bronze = spark.table(
    "workspace.banking_bronze.accounts"
)

print("Total records:", accounts_bronze.count())

In [0]:
%python
accounts_silver = accounts_bronze.dropDuplicates(
    ["account_id"]
)

print("Before:", accounts_bronze.count())
print("After:", accounts_silver.count())

In [0]:
%python
from pyspark.sql.functions import col, sum

accounts_silver.select(
    sum(col("account_id").isNull().cast("int")).alias("account_id_nulls"),
    sum(col("customer_id").isNull().cast("int")).alias("customer_id_nulls"),
    sum(col("branch_id").isNull().cast("int")).alias("branch_id_nulls"),
    sum(col("account_type").isNull().cast("int")).alias("account_type_nulls"),
    sum(col("opening_date").isNull().cast("int")).alias("opening_date_nulls"),
    sum(col("initial_balance").isNull().cast("int")).alias("balance_nulls")
).show()

In [0]:
%python
accounts_silver.groupBy("account_type").count().show()

In [0]:
%python
negative_balance_count = accounts_silver.filter(
    col("initial_balance") < 0
).count()

print("Negative balance records:", negative_balance_count)

In [0]:
%python
accounts_silver.select(
    "opening_date"
).describe().show()

In [0]:
%python
accounts_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.banking_silver.accounts"
    )

In [0]:
%python
spark.table(
    "workspace.banking_silver.accounts"
).count()

In [0]:
%python
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    to_date,
    to_timestamp,
    when,
    lit,
    row_number
)

from pyspark.sql.window import Window

In [0]:
%python
transactions_bronze = spark.table("banking_bronze.transactions")

print("Bronze Transactions Records:", transactions_bronze.count())

transactions_bronze.show(10)
transactions_bronze.printSchema()

In [0]:
%python
from pyspark.sql.functions import sum as spark_sum

transactions_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
%python
total_records = transactions_bronze.count()

unique_transactions = transactions_bronze.select(
    "transaction_id"
).distinct().count()

print("Total records:", total_records)
print("Unique transaction IDs:", unique_transactions)
print("Duplicate records:", total_records - unique_transactions)

In [0]:
%python
transactions_bronze.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_silver = transactions_bronze.dropDuplicates()

print("Records after duplicate removal:", transactions_silver.count())

In [0]:
%python
window_spec = Window.partitionBy("transaction_id").orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Records after transaction ID deduplication:",
      transactions_silver.count())

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", trim(col("transaction_type")))
)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "transaction_type",
    upper(trim(col("transaction_type")))
)

transactions_silver.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_silver.select("transaction_date").show(20, False)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "transaction_date",
    to_date(col("transaction_date"))
)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "amount",
    col("amount").cast("double")
)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "balance",
    col("balance").cast("double")
)

In [0]:
%python
transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

In [0]:
%python
transactions_silver = transactions_silver.filter(
    col("transaction_id").isNotNull()
)

In [0]:
%python
from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

transactions_bronze = spark.table("banking_bronze.transactions")

print("Columns:")
print(transactions_bronze.columns)

print("Schema:")
transactions_bronze.printSchema()

print("Total records:", transactions_bronze.count())

In [0]:
%python
transactions_silver = transactions_bronze.dropDuplicates()

print("Records after duplicate removal:",
      transactions_silver.count())

In [0]:
%python
window_spec = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Records after transaction ID deduplication:",
      transactions_silver.count())

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )
)

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn(
        "transaction_date",
        to_date(col("transaction_date"))
    )
    .withColumn(
        "amount",
        col("amount").cast("double")
    )
)

In [0]:
%python
transactions_silver.printSchema()

transactions_silver.show(10, False)

In [0]:
%python
from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

transactions_bronze = spark.table("banking_bronze.transactions")

print("Columns:", transactions_bronze.columns)
print("Total records:", transactions_bronze.count())

transactions_bronze.printSchema()

In [0]:
%python
total_records = transactions_bronze.count()

unique_transaction_ids = transactions_bronze \
    .select("transaction_id") \
    .distinct() \
    .count()

print("Total records:", total_records)
print("Unique transaction IDs:", unique_transaction_ids)
print("Duplicate transaction IDs:",
      total_records - unique_transaction_ids)

In [0]:
%python
from pyspark.sql.functions import sum as spark_sum

transactions_bronze.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
%python
transactions_bronze.groupBy("transaction_type") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_bronze.groupBy("payment_method") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_bronze.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_bronze.filter(
    col("amount") <= 0
).show(20, False)

In [0]:
%python
transactions_bronze.groupBy("transaction_type") \
    .agg(
        spark_sum("amount").alias("total_amount"),
        spark_sum(
            (col("amount") < 0).cast("int")
        ).alias("negative_count")
    ) \
    .orderBy("transaction_type") \
    .show()

In [0]:
%python
transactions_bronze.groupBy(
    "transaction_type"
).agg(
    spark_sum(
        (col("amount") > 0).cast("int")
    ).alias("positive_count"),
    
    spark_sum(
        (col("amount") < 0).cast("int")
    ).alias("negative_count"),
    
    spark_sum(
        (col("amount") == 0).cast("int")
    ).alias("zero_count")
).show()

In [0]:
%python
transactions_bronze.groupBy("status") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_bronze.groupBy("payment_method") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

In [0]:
%python
transactions_bronze.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_bronze.columns
]).show()

In [0]:
%python
transactions_bronze.filter(
    col("payment_method").isNull() |
    col("merchant_id").isNull()
).show(30, False)

In [0]:
%python
transactions_bronze.filter(
    col("payment_method").isNull()
).select(
    "transaction_id",
    "transaction_type",
    "amount",
    "payment_method",
    "merchant_id",
    "status"
).show(20, False)

In [0]:
%python
transactions_bronze.filter(
    col("merchant_id").isNull()
).select(
    "transaction_id",
    "transaction_type",
    "amount",
    "payment_method",
    "merchant_id",
    "status"
).show(20, False)

In [0]:
%python
transactions_silver = transactions_bronze


In [0]:
%python
transactions_silver = transactions_silver.dropDuplicates()

print(
    "Records after removing exact duplicates:",
    transactions_silver.count()
)

In [0]:
%python
window_spec = Window.partitionBy(
    "transaction_id"
).orderBy(
    col("transaction_date").desc()
)

transactions_silver = (
    transactions_silver
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(
    "Records after transaction ID deduplication:",
    transactions_silver.count()
)

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn(
        "transaction_id",
        trim(col("transaction_id"))
    )
    .withColumn(
        "account_id",
        trim(col("account_id"))
    )
    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )
    .withColumn(
        "payment_method",
        upper(trim(col("payment_method")))
    )
    .withColumn(
        "status",
        upper(trim(col("status")))
    )
)

In [0]:
%python
from pyspark.sql.functions import coalesce

transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

In [0]:
%python
transactions_silver.groupBy(
    "transaction_type"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
%python
transactions_silver.groupBy(
    "payment_method"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
%python
transactions_silver.groupBy(
    "status"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
%python
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
%python
transactions_silver.groupBy(
    "transaction_type"
).agg(
    spark_sum(
        (col("amount") > 0).cast("int")
    ).alias("positive_count"),
    
    spark_sum(
        (col("amount") < 0).cast("int")
    ).alias("negative_count"),
    
    spark_sum(
        (col("amount") == 0).cast("int")
    ).alias("zero_count")
).show()

In [0]:
%python
negative_count = transactions_silver.filter(
    col("amount") < 0
).count()

print("Negative amount records:", negative_count)

transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

print(
    "Records after removing negative amounts:",
    transactions_silver.count()
)

In [0]:
%python
transactions_silver.select(
    "amount"
).summary().show()

In [0]:
%python
transactions_silver.select(
    "transaction_date"
).summary().show()

In [0]:
%python
print(
    "Minimum transaction date:",
    transactions_silver.select("transaction_date")
    .agg({"transaction_date": "min"})
    .collect()[0][0]
)

print(
    "Maximum transaction date:",
    transactions_silver.select("transaction_date")
    .agg({"transaction_date": "max"})
    .collect()[0][0]
)

In [0]:
%python
print(
    "Total records:",
    transactions_silver.count()
)

print(
    "Unique transaction IDs:",
    transactions_silver
    .select("transaction_id")
    .distinct()
    .count()
)

In [0]:
%python
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
%python
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
%python
transactions_silver.orderBy(
    "transaction_date"
).show(20, False)

In [0]:
%python
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit,
    coalesce,
    sum as spark_sum
)

In [0]:
%python
transactions_bronze = spark.table("banking_bronze.transactions")

transactions_silver = transactions_bronze

print("Bronze records:", transactions_bronze.count())

In [0]:
%python
transactions_silver = transactions_silver.dropDuplicates()

print(
    "After exact duplicate removal:",
    transactions_silver.count()
)

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("status", upper(trim(col("status"))))
)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

In [0]:
%python
transactions_silver.filter(
    col("merchant_id") == "UNKNOWN"
).count()

In [0]:
%python
print(
    "Negative amount records:",
    transactions_silver.filter(
        col("amount") < 0
    ).count()
)

In [0]:
%python
from pyspark.sql.functions import col, trim, upper, lit, coalesce
from pyspark.sql.functions import sum as spark_sum

transactions_bronze = spark.table("banking_bronze.transactions")

transactions_silver = transactions_bronze

print("Bronze records:", transactions_bronze.count())
print("Silver DataFrame created successfully!")

In [0]:
%python
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("status", upper(trim(col("status"))))
)

In [0]:
%python
transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

print(
    "Unknown merchant IDs:",
    transactions_silver.filter(
        col("merchant_id") == "UNKNOWN"
    ).count()
)

In [0]:
%python
transactions_silver = transactions_silver.dropDuplicates()

print(
    "Records after duplicate removal:",
    transactions_silver.count()
)

In [0]:
%python
print(
    "Negative amounts:",
    transactions_silver.filter(
        col("amount") < 0
    ).count()
)

transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

print(
    "Records after amount cleaning:",
    transactions_silver.count()
)

In [0]:
%python
transactions_silver.printSchema()
transactions_silver.show(10, False)

In [0]:
%python
total_records = transactions_silver.count()

unique_ids = transactions_silver.select(
    "transaction_id"
).distinct().count()

print("Total Silver records:", total_records)
print("Unique transaction IDs:", unique_ids)
print("Duplicate transaction IDs:", total_records - unique_ids)

In [0]:
%python
transactions_silver.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in transactions_silver.columns
]).show()

In [0]:
%python
transactions_silver.groupBy(
    "status"
).count().orderBy(
    col("count").desc()
).show()

In [0]:
%python

from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lit,
    coalesce
)

# 1. Load Bronze
transactions_bronze = spark.table("banking_bronze.transactions")

print("Bronze records:", transactions_bronze.count())

# 2. Remove exact duplicate rows
transactions_silver = transactions_bronze.dropDuplicates()

# 3. Clean text columns
transactions_silver = (
    transactions_silver
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn(
        "transaction_type",
        upper(trim(col("transaction_type")))
    )
    .withColumn(
        "payment_method",
        upper(trim(col("payment_method")))
    )
    .withColumn(
        "status",
        upper(trim(col("status")))
    )
)

# 4. Handle missing merchant IDs
transactions_silver = transactions_silver.withColumn(
    "merchant_id",
    coalesce(
        trim(col("merchant_id")),
        lit("UNKNOWN")
    )
)

# 5. Remove invalid negative/zero amounts
transactions_silver = transactions_silver.filter(
    col("amount") > 0
)

# 6. Show result
print("Silver records:", transactions_silver.count())

transactions_silver.printSchema()

transactions_silver.show(10, False)

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date

path = "/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/transactions_raw.csv"

# Load raw transactions
transactions_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(path)
)

print("Bronze records:", transactions_bronze.count())

# Create Silver
transactions_silver = (
    transactions_bronze
    .dropDuplicates(["transaction_id"])
    .withColumn("transaction_id", trim(col("transaction_id")))
    .withColumn("account_id", trim(col("account_id")))
    .withColumn("transaction_type", upper(trim(col("transaction_type"))))
    .withColumn("payment_method", upper(trim(col("payment_method"))))
    .withColumn("status", upper(trim(col("status"))))
    .withColumn("merchant_id", trim(col("merchant_id")))
    .withColumn("transaction_date", to_date(col("transaction_date")))
    .filter(col("amount") > 0)
)

print("Silver records:", transactions_silver.count())

transactions_silver.show(10)

In [0]:
transactions_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.transactions_silver")

print("Silver table saved successfully!")

In [0]:
spark.sql("SHOW TABLES IN default").show(truncate=False)

In [0]:
transactions_silver = spark.table("default.transactions_silver")

print("Silver records:", transactions_silver.count())

transactions_silver.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

print("Duplicate transaction IDs:")
transactions_silver.groupBy("transaction_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("NULL value check:")
transactions_silver.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_silver.columns
]).show()

print("Invalid amounts:")
transactions_silver.filter(col("amount") <= 0).show()

print("Transaction status:")
transactions_silver.groupBy("status").count().show()

print("Transaction type:")
transactions_silver.groupBy("transaction_type").count().show()

print("Payment method:")
transactions_silver.groupBy("payment_method").count().show()

In [0]:
from pyspark.sql.functions import col, trim, upper, coalesce, lit

transactions_silver = (
    transactions_silver
    .withColumn(
        "payment_method",
        coalesce(
            upper(trim(col("payment_method"))),
            lit("UNKNOWN")
        )
    )
    .withColumn(
        "merchant_id",
        coalesce(
            trim(col("merchant_id")),
            lit("UNKNOWN")
        )
    )
)

print("Silver records:", transactions_silver.count())

transactions_silver.groupBy("payment_method").count().show()

In [0]:
transactions_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.transactions_silver")

In [0]:
transactions_silver = spark.table("default.transactions_silver")

print("Final Silver records:", transactions_silver.count())

In [0]:
spark.sql("SHOW TABLES IN default").show(truncate=False)

In [0]:
accounts_bronze = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/accounts_raw.csv")
)

print("Accounts Bronze records:", accounts_bronze.count())

accounts_bronze.show(10)
accounts_bronze.printSchema()

In [0]:
%sql
SHOW TABLES IN default;

In [0]:
accounts_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/accounts_raw.csv")

accounts_bronze.write.mode("overwrite").saveAsTable("default.accounts_bronze")

print("Accounts Bronze created")
print("Records:", accounts_bronze.count())

display(accounts_bronze.limit(10))

In [0]:
customers_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/customers_raw.csv")

customers_bronze.write.mode("overwrite").saveAsTable("default.customers_bronze")

print("Customers Bronze:", customers_bronze.count())

In [0]:
branches_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/branches_raw.csv")

branches_bronze.write.mode("overwrite").saveAsTable("default.branches_bronze")

print("Branches Bronze:", branches_bronze.count())

In [0]:
merchants_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/merchants_raw.csv")

merchants_bronze.write.mode("overwrite").saveAsTable("default.merchants_bronze")

print("Merchants Bronze:", merchants_bronze.count())

In [0]:
transactions_bronze = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/banking_raw/banking_project_dataset/banking_project_dataset/transactions_raw.csv")

transactions_bronze.write.mode("overwrite").saveAsTable("default.transactions_bronze")

print("Transactions Bronze:", transactions_bronze.count())

In [0]:
%sql
SHOW TABLES IN default;

In [0]:
accounts_bronze = spark.table("default.accounts_bronze")

print("Accounts Bronze records:", accounts_bronze.count())
accounts_bronze.printSchema()

display(accounts_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, count

print("Duplicate account IDs:")

accounts_bronze.groupBy("account_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null values:")

accounts_bronze.select([
    count(col(c)).alias(c) for c in accounts_bronze.columns
]).show()

In [0]:
from pyspark.sql.functions import col, trim, upper, when

accounts_silver = (
    accounts_bronze
    .dropDuplicates(["account_id"])
    .withColumn(
        "account_type",
        when(
            col("account_type").isNull() | (trim(col("account_type")) == ""),
            "UNKNOWN"
        ).otherwise(trim(col("account_type")))
    )
)

print("Accounts Silver records:", accounts_silver.count())

display(accounts_silver.limit(10))

In [0]:
accounts_silver.write \
    .mode("overwrite") \
    .saveAsTable("default.accounts_silver")

In [0]:
print("Total:", accounts_silver.count())
print("Unique account IDs:", accounts_silver.select("account_id").distinct().count())

accounts_silver.groupBy("account_type").count().show()

In [0]:
accounts_silver.groupBy("account_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
customers_bronze = spark.table("default.customers_bronze")

print("Customers Bronze records:", customers_bronze.count())
customers_bronze.printSchema()

display(customers_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, count

print("Duplicate customer IDs:")

customers_bronze.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null values:")

customers_bronze.select([
    count(col(c)).alias(c) for c in customers_bronze.columns
]).show()

In [0]:
from pyspark.sql.functions import col, trim, lower, when

customers_silver = (
    customers_bronze
    .dropDuplicates(["customer_id"])
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn(
        "email",
        when(
            col("email").isNull() | (trim(col("email")) == ""),
            "unknown@email.com"
        ).otherwise(lower(trim(col("email"))))
    )
    .withColumn(
        "city",
        when(
            col("city").isNull() | (trim(col("city")) == ""),
            "Unknown"
        ).otherwise(trim(col("city")))
    )
    .withColumn("state", trim(col("state")))
)

In [0]:
print("Customers Silver records:", customers_silver.count())

customers_silver.select([
    count(col(c)).alias(c) for c in customers_silver.columns
]).show()

display(customers_silver.limit(10))

In [0]:
customers_silver.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
customers_silver.write \
    .mode("overwrite") \
    .saveAsTable("default.customers_silver")

In [0]:
branches_bronze = spark.table("default.branches_bronze")

print("Branches Bronze records:", branches_bronze.count())
branches_bronze.printSchema()

display(branches_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, count

print("Duplicate branch IDs:")

branches_bronze.groupBy("branch_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null values:")

branches_bronze.select([
    count(col(c)).alias(c) for c in branches_bronze.columns
]).show()

In [0]:
from pyspark.sql.functions import col, trim

branches_silver = (
    branches_bronze
    .dropDuplicates(["branch_id"])
    .withColumn("branch_id", trim(col("branch_id")))
    .withColumn("branch_name", trim(col("branch_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
)

print("Branches Silver records:", branches_silver.count())

display(branches_silver)

In [0]:
branches_silver.groupBy("branch_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
branches_silver.write \
    .mode("overwrite") \
    .saveAsTable("default.branches_silver")

In [0]:
merchants_bronze = spark.table("default.merchants_bronze")

print("Merchants Bronze records:", merchants_bronze.count())

merchants_bronze.printSchema()

display(merchants_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, count

print("Duplicate merchant IDs:")

merchants_bronze.groupBy("merchant_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("Null values:")

merchants_bronze.select([
    count(col(c)).alias(c) for c in merchants_bronze.columns
]).show()

In [0]:
from pyspark.sql.functions import col, trim, upper

merchants_silver = (
    merchants_bronze
    .dropDuplicates(["merchant_id"])
    .withColumn("merchant_id", trim(col("merchant_id")))
    .withColumn("merchant_name", trim(col("merchant_name")))
    .withColumn("merchant_category", upper(trim(col("merchant_category"))))
)

print("Merchants Silver records:", merchants_silver.count())

display(merchants_silver.limit(10))

In [0]:
merchants_silver.groupBy("merchant_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
merchants_silver.groupBy("merchant_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
merchants_silver.write \
    .mode("overwrite") \
    .saveAsTable("default.merchants_silver")

In [0]:
from pyspark.sql.functions import col, trim, upper

# Load Bronze table
merchants_bronze = spark.table("default.merchants_bronze")

# Clean and transform
merchants_silver = (
    merchants_bronze
    .dropDuplicates(["merchant_id"])
    .withColumn("merchant_id", trim(col("merchant_id")))
    .withColumn("merchant_name", trim(col("merchant_name")))
    .withColumn("merchant_category", upper(trim(col("merchant_category"))))
)

# Check record count
print("Merchants Bronze:", merchants_bronze.count())
print("Merchants Silver:", merchants_silver.count())

# Display sample
display(merchants_silver.limit(10))

In [0]:
merchants_silver.write \
    .mode("overwrite") \
    .saveAsTable("default.merchants_silver")

print("merchants_silver saved successfully")

In [0]:
%sql
SHOW TABLES IN default;

In [0]:
# Load all Silver tables
accounts_silver = spark.table("default.accounts_silver")
customers_silver = spark.table("default.customers_silver")
branches_silver = spark.table("default.branches_silver")
merchants_silver = spark.table("default.merchants_silver")
transactions_silver = spark.table("default.transactions_silver")

print("========== SILVER RECORD COUNTS ==========")

print("Accounts     :", accounts_silver.count())
print("Customers    :", customers_silver.count())
print("Branches     :", branches_silver.count())
print("Merchants    :", merchants_silver.count())
print("Transactions :", transactions_silver.count())

In [0]:
from pyspark.sql.functions import col, count

print("========== TRANSACTION QUALITY ==========")

print("Duplicate transaction IDs:")
transactions_silver.groupBy("transaction_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

print("NULL values:")
transactions_silver.select([
    count(col(c)).alias(c) for c in transactions_silver.columns
]).show()

print("Transaction status:")
transactions_silver.groupBy("status").count().show()

print("Transaction type:")
transactions_silver.groupBy("transaction_type").count().show()

In [0]:
from pyspark.sql.functions import col, trim, upper

branches_silver = (
    spark.table("banking_bronze.branches")
    .select(
        trim(col("branch_id")).alias("branch_id"),
        trim(col("branch_name")).alias("branch_name"),
        trim(col("city")).alias("city"),
        trim(col("state")).alias("state")
    )
    .filter(col("branch_id").isNotNull())
    .dropDuplicates(["branch_id"])
)

In [0]:
print("Bronze records:", spark.table("banking_bronze.branches").count())
print("Silver records:", branches_silver.count())

print("Duplicate branch IDs after cleaning:")

branches_silver.groupBy("branch_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
branches_silver.write \
    .mode("overwrite") \
    .saveAsTable("banking_silver.branches")

In [0]:
spark.sql("""
SHOW TABLES IN banking_silver
""").show()

In [0]:
from pyspark.sql.functions import col, trim

merchants_silver = (
    spark.table("banking_bronze.merchants")
    .select(
        trim(col("merchant_id")).alias("merchant_id"),
        trim(col("merchant_name")).alias("merchant_name"),
        trim(col("merchant_category")).alias("merchant_category")
    )
    .filter(col("merchant_id").isNotNull())
    .dropDuplicates(["merchant_id"])
)

In [0]:
print("Bronze merchants:", spark.table("banking_bronze.merchants").count())
print("Silver merchants:", merchants_silver.count())

In [0]:
merchants_silver.groupBy("merchant_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
merchants_silver.write \
    .mode("overwrite") \
    .saveAsTable("banking_silver.merchants")

In [0]:
spark.sql("""
SHOW TABLES IN banking_silver
""").show()

In [0]:
spark.sql("SHOW TABLES IN default").show(100, False)

In [0]:
transactions = spark.table("default.transactions_silver")
accounts = spark.table("default.accounts_silver")
customers = spark.table("default.customers_silver")
merchants = spark.table("default.merchants_silver")
branches = spark.table("default.branches_silver")

In [0]:
gold_transactions = (
    transactions
    .join(accounts, "account_id", "left")
    .join(customers, "customer_id", "left")
    .join(merchants, "merchant_id", "left")
    .join(branches, "branch_id", "left")
)

display(gold_transactions.limit(10))